# Aridity Index (AI)
Extract mean aridity index within a 10 km bounding box for each site.

Source: Zomer et al. (2022) Global Aridity Index v3.1
        https://doi.org/10.6084/m9.figshare.7504448

- Using Annual ETO version (ai_v31_yr.tif) which represents the mean annual aridity index.
- Raw values are stored as uint16 integers multiplied by 10,000.
- Divide by 10,000 to get the actual aridity index.
- Higher values = more humid; lower values = more arid.


### AI for all combined 1, 4, 16, and 100 km^2 

In [6]:
# # Libraries
# import numpy as np
# import pandas as pd
# import rasterio
# from rasterio.windows import from_bounds

# # Retrieving CSV and TIF from Cyberduck
# SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
# ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"   # path to your local Global-AI_ET0_annual_v3_1 TIF
# # CSV being saved in CyberDuck
# OUTPUT_CSV = "/capstone/aridgw/data/sites_aridity_index.csv"

# # Bounding boxes to extract (in km^2)
# BBOX_SIZES_KM = [1, 2, 4, 10]  # side lengths (km) producing areas of 1, 4, 16, 100 km²


# AI_SCALE = 10_000.0  # raw values are AI * 10000


# def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
#     """Convert km offset to degrees lat/lon at a given latitude."""
#     lat_deg = km / 111.0
#     lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
#     return lat_deg, lon_deg
 
 
# def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float, bbox_km: float) -> float:
#     """Extract mean aridity index within bbox_km around a point."""
#     half = bbox_km / 2
#     lat_off, lon_off = km_to_deg(half, lat)
 
#     window = from_bounds(
#         left=lon - lon_off,
#         bottom=lat - lat_off,
#         right=lon + lon_off,
#         top=lat + lat_off,
#         transform=src.transform
#     )
#     data = src.read(1, window=window).astype(float)
#     data[data == 0] = np.nan  # 0 is nodata in this dataset
 
#     if np.all(np.isnan(data)):
#         return np.nan
 
#     return float(np.nanmean(data)) / AI_SCALE
 
 
# def main():
#     df = pd.read_csv(SITES_CSV)
#     sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()
 
#     print(f"Extracting aridity for {len(sites)} sites at bbox sizes: {BBOX_SIZES_KM} km...")
 
#     AREA_LABELS = {1: "1km2", 2: "4km2", 4: "16km2", 10: "100km2"}
 
#     with rasterio.open(ARIDITY_TIF) as src:
#         for km in BBOX_SIZES_KM:
#             col = f"AI_{AREA_LABELS[km]}"
#             sites[col] = sites.apply(
#                 lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"], km),
#                 axis=1
#             )
#             print(f"  {AREA_LABELS[km]} done")
 
#     ai_cols = [f"AI_{AREA_LABELS[km]}" for km in BBOX_SIZES_KM]
#     result = df.merge(sites[["site_id"] + ai_cols], on="site_id", how="left")
#     result.to_csv(OUTPUT_CSV, index=False)
 
#  # Uncomment to save onto Cyberduck
#    # print(f"\nSaved: {OUTPUT_CSV}")
# # Print results
#     #print(result[["site_id", "latitude", "longitude"] + ai_cols].drop_duplicates().to_string())
 
 
# if __name__ == "__main__":
#     main()
 

# AI for 1 km^2 

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# Retrieving CSV and TIF from Cyberduck
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
# CSV being saved in CyberDuck - Using Path
OUTPUT_CSV = "/capstone/aridgw/data/site_data/aridity_index_sites/sites_aridity_1km2.csv"
# Size of Bounding Box
BBOX_KM = 1    # side length (km) producing 1km2 area
AREA_LABEL = "1km2"
AI_SCALE = 10_000.0   # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float) -> float:
    """Extract mean aridity index within BBOX_KM around a point."""
    half = BBOX_KM / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting AI_{AREA_LABEL} for {len(sites)} sites...")

    with rasterio.open(ARIDITY_TIF) as src:
        sites[f"AI_{AREA_LABEL}"] = sites.apply(
            lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"]),
            axis=1
        )

    result = df.merge(sites[["site_id", f"AI_{AREA_LABEL}"]], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"Saved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude", f"AI_{AREA_LABEL}"]].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

Extracting AI_1km2 for 50 sites...
Saved: /capstone/aridgw/data/sites_aridity_1km2.csv
                    site_id   latitude   longitude  AI_1km2
0      KSGS.371852100505801  37.315020 -100.850500   0.2871
21     KSGS.372043101363101  37.344950 -101.610400   0.2302
42     KSGS.372539100142504  37.429490 -100.243400   0.3153
63     KSGS.373331098033301  37.560960  -98.058000   0.4661
84     KSGS.373607100565301  37.598740 -100.949700   0.2782
105    KSGS.374111099070401  37.684480  -99.119250   0.3797
126    KSGS.374125100344101  37.690240 -100.579500   0.3101
147    KSGS.374747100552101  37.794920 -100.922500   0.2908
168    KSGS.375145100485701  37.862090 -100.817900   0.2917
189    KSGS.375454101075401  37.915720 -101.130600   0.2784
210            TWDB.0753701  35.160000 -102.488600   0.2411
231            TWDB.1005225  34.970000 -102.438100   0.2388
252            TWDB.1038101  34.460000 -102.352200   0.2412
273            TWDB.1039702  34.380000 -102.233600   0.2436
294          

# AI for 4 km^2 

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# Retrieving CSV and TIF from Cyberduck
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
# CSV being saved in CyberDuck - Using Path
OUTPUT_CSV = "/capstone/aridgw/data/site_data/aridity_index_sites/sites_aridity_4km2.csv"
# Bounding Box
BBOX_KM = 2        # side length (km) producing 4km2 area
AREA_LABEL = "4km2"
AI_SCALE = 10_000.0   # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float) -> float:
    """Extract mean aridity index within BBOX_KM around a point."""
    half = BBOX_KM / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting AI_{AREA_LABEL} for {len(sites)} sites...")

    with rasterio.open(ARIDITY_TIF) as src:
        sites[f"AI_{AREA_LABEL}"] = sites.apply(
            lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"]),
            axis=1
        )

    result = df.merge(sites[["site_id", f"AI_{AREA_LABEL}"]], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"Saved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude", f"AI_{AREA_LABEL}"]].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

Extracting AI_4km2 for 50 sites...
Saved: /capstone/aridgw/data/sites_aridity_4km2.csv
                    site_id   latitude   longitude   AI_4km2
0      KSGS.371852100505801  37.315020 -100.850500  0.288667
21     KSGS.372043101363101  37.344950 -101.610400  0.230800
42     KSGS.372539100142504  37.429490 -100.243400  0.314567
63     KSGS.373331098033301  37.560960  -98.058000  0.466383
84     KSGS.373607100565301  37.598740 -100.949700  0.278783
105    KSGS.374111099070401  37.684480  -99.119250  0.380217
126    KSGS.374125100344101  37.690240 -100.579500  0.309900
147    KSGS.374747100552101  37.794920 -100.922500  0.290283
168    KSGS.375145100485701  37.862090 -100.817900  0.292417
189    KSGS.375454101075401  37.915720 -101.130600  0.278600
210            TWDB.0753701  35.160000 -102.488600  0.240800
231            TWDB.1005225  34.970000 -102.438100  0.238567
252            TWDB.1038101  34.460000 -102.352200  0.240000
273            TWDB.1039702  34.380000 -102.233600  0.24378

# AI for 16 km^2 

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# Retrieving CSV and TIF from Cyberduck
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
# CSV being saved in CyberDuck - Using Path
OUTPUT_CSV = "/capstone/aridgw/data/site_data/aridity_index_sites/sites_aridity_16km2.csv"
# Bounding box
BBOX_KM = 4        # side length (km) producing 16km2 area
AREA_LABEL = "16km2"
AI_SCALE = 10_000.0   # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float) -> float:
    """Extract mean aridity index within BBOX_KM around a point."""
    half = BBOX_KM / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting AI_{AREA_LABEL} for {len(sites)} sites...")

    with rasterio.open(ARIDITY_TIF) as src:
        sites[f"AI_{AREA_LABEL}"] = sites.apply(
            lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"]),
            axis=1
        )

    result = df.merge(sites[["site_id", f"AI_{AREA_LABEL}"]], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"Saved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude", f"AI_{AREA_LABEL}"]].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

Extracting AI_16km2 for 50 sites...
Saved: /capstone/aridgw/data/sites_aridity_16km2.csv
                    site_id   latitude   longitude  AI_16km2
0      KSGS.371852100505801  37.315020 -100.850500  0.288970
21     KSGS.372043101363101  37.344950 -101.610400  0.230040
42     KSGS.372539100142504  37.429490 -100.243400  0.314155
63     KSGS.373331098033301  37.560960  -98.058000  0.465795
84     KSGS.373607100565301  37.598740 -100.949700  0.280025
105    KSGS.374111099070401  37.684480  -99.119250  0.380690
126    KSGS.374125100344101  37.690240 -100.579500  0.310110
147    KSGS.374747100552101  37.794920 -100.922500  0.289180
168    KSGS.375145100485701  37.862090 -100.817900  0.292335
189    KSGS.375454101075401  37.915720 -101.130600  0.278805
210            TWDB.0753701  35.160000 -102.488600  0.240845
231            TWDB.1005225  34.970000 -102.438100  0.238905
252            TWDB.1038101  34.460000 -102.352200  0.240270
273            TWDB.1039702  34.380000 -102.233600  0.243

# AI for 100 km^2 

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# Retrieving CSV and TIF from Cyberduck
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"
# CSV being saved in CyberDuck - Using Path
OUTPUT_CSV = "/capstone/aridgw/data/site_data/aridity_index_sites/sites_aridity_100km2.csv"
# Bounding Box
BBOX_KM = 10        # side length (km) producing 100km2 area
AREA_LABEL = "100km2"
AI_SCALE = 10_000.0   # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float) -> float:
    """Extract mean aridity index within BBOX_KM around a point."""
    half = BBOX_KM / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting AI_{AREA_LABEL} for {len(sites)} sites...")

    with rasterio.open(ARIDITY_TIF) as src:
        sites[f"AI_{AREA_LABEL}"] = sites.apply(
            lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"]),
            axis=1
        )

    result = df.merge(sites[["site_id", f"AI_{AREA_LABEL}"]], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"Saved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude", f"AI_{AREA_LABEL}"]].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

Extracting AI_100km2 for 50 sites...
Saved: /capstone/aridgw/data/sites_aridity_100km2.csv
                    site_id   latitude   longitude  AI_100km2
0      KSGS.371852100505801  37.315020 -100.850500   0.287641
21     KSGS.372043101363101  37.344950 -101.610400   0.229530
42     KSGS.372539100142504  37.429490 -100.243400   0.313350
63     KSGS.373331098033301  37.560960  -98.058000   0.465467
84     KSGS.373607100565301  37.598740 -100.949700   0.280066
105    KSGS.374111099070401  37.684480  -99.119250   0.380786
126    KSGS.374125100344101  37.690240 -100.579500   0.310267
147    KSGS.374747100552101  37.794920 -100.922500   0.288092
168    KSGS.375145100485701  37.862090 -100.817900   0.291765
189    KSGS.375454101075401  37.915720 -101.130600   0.278916
210            TWDB.0753701  35.160000 -102.488600   0.242202
231            TWDB.1005225  34.970000 -102.438100   0.238882
252            TWDB.1038101  34.460000 -102.352200   0.240197
273            TWDB.1039702  34.380000 -1

# Spearman Rank Correlation for AI